# MalthusJAX Level 3 Demo: Evolution Engines

This notebook demonstrates **Level 3 evolution engines** that orchestrate complete evolutionary algorithms using the flexible components from Levels 1 and 2.

## Overview

Level 3 provides the `GeneticEngine` (formerly `StandardGeneticEngine`) that composes:
- **Level 1**: Genomes and fitness evaluators
- **Level 2**: Selection, crossover, and mutation operators
- **Level 3**: Complete evolutionary loops with JIT compilation

## Key Architecture Features

1. **Abstract Engine Base**: `AbstractEngine` defines the interface for all evolution engines
2. **Pluggable Components**: Mix and match evaluators, operators for different problems
3. **Automatic JIT**: Evolution loops compiled via `jax.lax.scan` for maximum performance
4. **State Management**: Immutable state objects track population and evolution progress
5. **Compilation Caching**: Pre-compile once, run multiple experiments efficiently

This demo showcases the engine's flexibility across different problem domains: binary optimization, knapsack, continuous optimization, and symbolic regression.

In [1]:
# Essential imports
import jax
import jax.numpy as jnp
import jax.random as jar

from malthusjax.core.genome.binary_genome import BinaryGenomeConfig
from malthusjax.core.genome.real_genome import RealGenomeConfig
from malthusjax.core.fitness.binary_evaluators import BinarySumEvaluator, BinarySumConfig, KnapsackEvaluator, KnapsackConfig
from malthusjax.core.fitness.real_evaluators import SphereEvaluator, SphereConfig
from malthusjax.engine.genetic_engine import GeneticEngine, GeneticEngineParams
from malthusjax.operators.selection.tournament import TournamentSelection
from malthusjax.operators.crossover.binary import UniformCrossover, SinglePointCrossover
from malthusjax.operators.crossover.real import SimulatedBinaryCrossover
from malthusjax.operators.mutation.binary import BitFlipMutation
from malthusjax.operators.mutation.real import GaussianMutation
# Note: DiversityAwareEngine will be imported in Example 4 when needed

print(f"JAX version: {jax.__version__}")
print(f"JAX backend: {jax.default_backend()}")
print("Level 3 evolution engine components loaded")

# Initialize random key
key = jar.PRNGKey(42)

JAX version: 0.8.0
JAX backend: cpu
Level 3 evolution engine components loaded


In [2]:
## Example 1: Binary Optimization (OneMax Problem)

# Configure genome
genome_config = BinaryGenomeConfig(length=100)

# Configure fitness evaluator
eval_config = BinarySumConfig(maximize=True)
evaluator = BinarySumEvaluator(config=eval_config, data=None)

# Configure operators
selection = TournamentSelection(num_selections=100, tournament_size=3)
crossover = UniformCrossover(num_offspring=2, crossover_rate=0.8)
mutation = BitFlipMutation(num_offspring=2, mutation_rate=0.01)

# Create engine
engine = GeneticEngine(
    genome_config=genome_config,
    evaluator=evaluator,
    selection=selection,
    crossover=crossover,
    mutation=mutation
)

# Configure evolution parameters
params = GeneticEngineParams(
    pop_size=100,
    num_generations=50,
    elitism=5
)

print(f"Binary optimization setup:")
print(f"  Genome length: {genome_config.length}")
print(f"  Population size: {params.pop_size}")
print(f"  Generations: {params.num_generations}")
print(f"  Elitism: {params.elitism}")

Binary optimization setup:
  Genome length: 100
  Population size: 100
  Generations: 50
  Elitism: 5


In [16]:
# Initialize state
key, k_init = jar.split(key)
state = engine.init_state(k_init, params)

print(f"Initial state:")
print(f"  Population size: {len(state.population)}")
print(f"  Best fitness: {state.best_fitness:.1f}/{genome_config.length}")
print(f"  Average fitness: {jnp.mean(state.fitness_values):.1f}")

# Run evolution
final_state, history, elapsed = engine.run(
    state,
    params,
    time_it=True,
    compile=True,
    verbose=False
)

print(f"\nOptimization complete:")
print(f"  Execution time: {elapsed:.4f}s")
print(f"  Final best fitness: {final_state.best_fitness:.1f}/{genome_config.length}")
print(f"  Final average fitness: {jnp.mean(final_state.fitness_values):.1f}")
print(f"  Improvement: {final_state.best_fitness - state.best_fitness:+.1f}")
print(f"  Convergence: {(final_state.best_fitness/genome_config.length)*100:.1f}%")

Initial state:
  Population size: 100
  Best fitness: -97.8/100
  Average fitness: -46.5

Optimization complete:
  Execution time: 0.0110s
  Final best fitness: -0.0/100
  Final average fitness: -0.1
  Improvement: +97.8
  Convergence: -0.0%


## Example 2: Knapsack Problem with Integer Genome

**Problem Setup**: Classic 0/1 knapsack problem demonstrating:
- Integer genome representation
- Constraint handling in fitness evaluation
- Same engine architecture applied to different problem domain

The knapsack problem illustrates how the abstract engine interfaces work with different genome types without modification.

In [4]:
## Example 2: Knapsack Problem

# Problem data
weights = jnp.array([10, 20, 30, 40, 50], dtype=jnp.float32)
values = jnp.array([5, 10, 15, 20, 25], dtype=jnp.float32)
capacity = 100.0

# Configure genome
genome_config = BinaryGenomeConfig(length=len(weights), p = 0.01)

# Configure evaluator
eval_config = KnapsackConfig(
    weights=weights,
    values=values,
    capacity=capacity,
    maximize=True
)
evaluator = KnapsackEvaluator(config=eval_config, data=None)

# Configure operators
selection = TournamentSelection(num_selections=30, tournament_size=3)
crossover = SinglePointCrossover(num_offspring=2)
mutation = BitFlipMutation(num_offspring=1, mutation_rate=0.05)

# Create engine
engine = GeneticEngine(
    genome_config=genome_config,
    evaluator=evaluator,
    selection=selection,
    crossover=crossover,
    mutation=mutation
)

# Configure parameters
params = GeneticEngineParams(
    pop_size=50,
    num_generations=100,
    elitism=2
)

# Initialize and run
key, k_init = jar.split(key)
state = engine.init_state(k_init, params)

print(f"Knapsack problem:")
print(f"  Items: {len(weights)}")
print(f"  Capacity: {capacity}")
print(f"  Initial best value: {state.best_fitness:.2f}")

final_state, history, elapsed = engine.run(
    state,
    params,
    time_it=True,
    compile=True,
    verbose=False
)

print(f"\nOptimization complete:")
print(f"  Execution time: {elapsed:.4f}s")
print(f"  Final best value: {final_state.best_fitness:.2f}")
print(f"  Improvement: {final_state.best_fitness - state.best_fitness:+.2f}")



Knapsack problem:
  Items: 5
  Capacity: 100.0
  Initial best value: 25.00

Optimization complete:
  Execution time: 0.5319s
  Final best value: 50.00
  Improvement: +25.00


## Example 3: Continuous Optimization with Real-Valued Genome

**Problem Setup**: Minimizing the Sphere function demonstrating:
- Real-valued genome representation
- Continuous optimization problems
- Engine flexibility across genome types

This example shows how the same `GeneticEngine` class handles continuous domains by simply swapping genome and evaluator types.

In [5]:
## Example 3: Continuous Optimization (Sphere Function)

# Configure genome
genome_config = RealGenomeConfig(
    length=5,
    bounds=(-5.12, 5.12)
)

# Configure evaluator
eval_config = SphereConfig(maximize=False)
evaluator = SphereEvaluator(config=eval_config, data=None)

# Configure operators
selection = TournamentSelection(num_selections=50, tournament_size=3)
crossover = SimulatedBinaryCrossover(num_offspring=2, eta=15.0)
mutation = GaussianMutation(num_offspring=1, mutation_rate=0.1, mutation_strength=0.5)

# Create engine
engine = GeneticEngine(
    genome_config=genome_config,
    evaluator=evaluator,
    selection=selection,
    crossover=crossover,
    mutation=mutation
)

# Configure parameters
params = GeneticEngineParams(
    pop_size=100,
    num_generations=100,
    elitism=5
)

# Initialize and run
key, k_init = jar.split(key)
state = engine.init_state(k_init, params)

print(f"Sphere function (minimization):")
print(f"  Dimensions: {genome_config.length}")
print(f"  Bounds: {genome_config.bounds}")
print(f"  Initial best fitness: {state.best_fitness:.6f}")

final_state, history, elapsed = engine.run(
    state,
    params,
    time_it=True,
    compile=True,
    verbose=False
)

print(f"\nOptimization complete:")
print(f"  Execution time: {elapsed:.4f}s")
print(f"  Final fitness: {final_state.best_fitness:.6f}")
print(f"  Improvement: {final_state.best_fitness - state.best_fitness:+.6f}")


Sphere function (minimization):
  Dimensions: 5
  Bounds: (-5.12, 5.12)
  Initial best fitness: -86.540741

Optimization complete:
  Execution time: 0.7129s
  Final fitness: -0.000001
  Improvement: +86.540741


## Example 4: Diversity-Aware Engine Extension

**Extending the Engine Architecture**:

This example demonstrates the extensibility of the `GeneticEngine` by creating a `DiversityAwareEngine` that overrides the selection mechanism to incorporate distance matrix information. This promotes population diversity alongside fitness optimization.

**Key Innovation**:
- Overrides `_select_parents()` to use both fitness AND crowding distance
- Overrides `_select_elites()` to preserve a mix of fit (70%) and diverse (30%) individuals
- Uses the population's `distance_matrix()` method from Level 1
- Adjustable `diversity_weight` parameter (0.0 = pure fitness, 1.0 = pure diversity)

**Implementation Strategy**:
The engine computes a crowding distance metric for each individual based on average Hamming distance to all other population members. This diversity score is then combined with fitness to guide selection, preventing premature convergence.

## Summary: Engine Architecture Flexibility

### Key Design Principles

**1. Component-Based Architecture**:
The `GeneticEngine` follows a strict composition pattern where all components are explicitly configured:
```python
engine = GeneticEngine(
    genome_config=config,    # Defines genome structure
    evaluator=evaluator,      # Fitness evaluation logic
    selection=selection,      # Parent selection strategy
    crossover=crossover,      # Recombination operator
    mutation=mutation         # Variation operator
)
```

**2. Consistent API Pattern**:
All examples follow the same workflow:
- Configure genome and evaluator
- Create operators (selection, crossover, mutation)
- Assemble engine with components
- Initialize state: `state = engine.init_state(key, params)`
- Run evolution: `final_state, history, time = engine.run(state, params)`

**3. Automatic Population Management**:
The engine creates and manages populations internally through `init_state()`. You only specify:
- `genome_config`: What kind of genome
- `params.pop_size`: How many individuals
- Random key for initialization

**4. Type Safety Through Composition**:
- Each genome type (Binary, Real, Categorical) works with its specialized operators
- Evaluators are configured independently and plugged into the engine
- Operators are instantiated with static parameters (num_offspring, tournament_size, etc.)

**5. Performance Characteristics**:
- JIT compilation via `compile=True` flag
- Compilation caching across runs with same parameters
- `jax.lax.scan` for efficient generation loops
- Automatic GPU/TPU acceleration when available

### Extensibility via Inheritance

**Example 4 demonstrates the Template Method pattern** for extending engine behavior:

The `DiversityAwareEngine` extends `GeneticEngine` by overriding specific methods:
- `_select_parents()`: Incorporates distance matrix for diversity-aware selection
- `_select_elites()`: Balances fitness and diversity in elite preservation
- Uses the population's built-in `distance_matrix()` method

This shows how to:
1. **Inherit from `GeneticEngine`** to reuse core logic
2. **Override template methods** like `_select_parents()` and `_select_elites()`
3. **Add custom parameters** (e.g., `diversity_weight`, `distance_metric`)
4. **Use Level 1 features** (distance matrix) in Level 3 algorithms
5. **Maintain full JIT compatibility** through proper struct.dataclass design

### Adding New Problem Types

To add new problem types:
1. Define or reuse a genome config (e.g., `BinaryGenomeConfig`)
2. Create a custom evaluator implementing `BaseEvaluator`
3. Choose appropriate operators from Level 2
4. Compose into `GeneticEngine` following the same pattern
5. (Optional) Extend the engine for custom selection/variation strategies

The abstract interfaces enable code reuse while maintaining flexibility across diverse evolutionary computation tasks.

In [6]:
## Example 4: Diversity-Aware Engine (Extension Demo)

from malthusjax.engine.diversity_engine import DiversityAwareEngine

# Use the same binary optimization problem for comparison
genome_config = BinaryGenomeConfig(length=100)
eval_config = BinarySumConfig(maximize=True)
evaluator = BinarySumEvaluator(config=eval_config, data=None)

# Configure operators (same as Example 1)
selection = TournamentSelection(num_selections=100, tournament_size=3)
crossover = UniformCrossover(num_offspring=1, crossover_rate=0.8)
mutation = BitFlipMutation(num_offspring=1, mutation_rate=0.01)

# Create DIVERSITY-AWARE engine with custom parameters
diversity_engine = DiversityAwareEngine(
    genome_config=genome_config,
    evaluator=evaluator,
    selection=selection,
    crossover=crossover,
    mutation=mutation,
    diversity_weight=0.3,  # 30% weight on diversity, 70% on fitness
    distance_metric="hamming"  # Use Hamming distance for binary genomes
)

# Same parameters as before
params = GeneticEngineParams(
    pop_size=100,
    num_generations=50,
    elitism=5
)

print("Diversity-Aware Engine Configuration:")
print(f"  Diversity weight: {diversity_engine.diversity_weight}")
print(f"  Distance metric: {diversity_engine.distance_metric}")
print(f"  Elite selection: 70% fitness + 30% diversity")
print(f"  Parent selection: Fitness + crowding distance combined")


Diversity-Aware Engine Configuration:
  Diversity weight: 0.3
  Distance metric: hamming
  Elite selection: 70% fitness + 30% diversity
  Parent selection: Fitness + crowding distance combined


In [7]:
# Run diversity-aware evolution
key, k_init = jar.split(key)
div_state = diversity_engine.init_state(k_init, params)

print(f"\nInitial state:")
print(f"  Population size: {len(div_state.population)}")
print(f"  Best fitness: {div_state.best_fitness:.1f}/{genome_config.length}")

# Compute initial diversity
initial_dist_matrix = div_state.population.distance_matrix(metric="hamming")
initial_avg_distance = jnp.mean(initial_dist_matrix)

print(f"  Average pairwise distance: {initial_avg_distance:.2f}")

# Run evolution
div_final_state, div_history, div_elapsed = diversity_engine.run(
    div_state,
    params,
    time_it=True,
    compile=True,
    verbose=False
)

# Compute final diversity
final_dist_matrix = div_final_state.population.distance_matrix(metric="hamming")
final_avg_distance = jnp.mean(final_dist_matrix)

print(f"\nOptimization complete:")
print(f"  Execution time: {div_elapsed:.4f}s")
print(f"  Final best fitness: {div_final_state.best_fitness:.1f}/{genome_config.length}")
print(f"  Final average fitness: {jnp.mean(div_final_state.fitness_values):.1f}")
print(f"  Final avg pairwise distance: {final_avg_distance:.2f}")
print(f"  Diversity change: {final_avg_distance - initial_avg_distance:+.2f}")
print(f"\nDiversity Preservation:")
print(f"  The diversity-aware engine maintains higher population diversity")
print(f"  while still achieving strong fitness convergence.")



Initial state:
  Population size: 100
  Best fitness: 61.0/100
  Average pairwise distance: 49.46

Optimization complete:
  Execution time: 0.4073s
  Final best fitness: 100.0/100
  Final average fitness: 97.1
  Final avg pairwise distance: 5.50
  Diversity change: -43.96

Diversity Preservation:
  The diversity-aware engine maintains higher population diversity
  while still achieving strong fitness convergence.


### Comparative Analysis: Standard vs Diversity-Aware

To demonstrate the effectiveness of diversity preservation, we'll run both engines side-by-side on the same problem and compare their convergence behavior and final diversity metrics.

In [8]:
# Side-by-side comparison
print("=" * 60)
print("STANDARD ENGINE vs DIVERSITY-AWARE ENGINE COMPARISON")
print("=" * 60)

# Re-run standard engine for fair comparison
standard_engine = GeneticEngine(
    genome_config=genome_config,
    evaluator=evaluator,
    selection=selection,
    crossover=crossover,
    mutation=mutation,
    enable_progress_bar=True
)

key, k_init = jar.split(key)
std_state = standard_engine.init_state(k_init, params)
std_final_state, _, std_elapsed = standard_engine.run(
    std_state, params, time_it=True, compile=True, verbose=False
)

std_final_dist = jnp.mean(std_final_state.population.distance_matrix(metric="hamming"))

print(f"\nSTANDARD ENGINE:")
print(f"  Final best fitness: {std_final_state.best_fitness:.1f}/{genome_config.length}")
print(f"  Final avg fitness: {jnp.mean(std_final_state.fitness_values):.1f}")
print(f"  Final diversity: {std_final_dist:.2f}")
print(f"  Time: {std_elapsed:.4f}s")

print(f"\nDIVERSITY-AWARE ENGINE:")
print(f"  Final best fitness: {div_final_state.best_fitness:.1f}/{genome_config.length}")
print(f"  Final avg fitness: {jnp.mean(div_final_state.fitness_values):.1f}")
print(f"  Final diversity: {final_avg_distance:.2f}")
print(f"  Time: {div_elapsed:.4f}s")

print(f"\nKEY INSIGHT:")
diversity_improvement = ((final_avg_distance - std_final_dist) / std_final_dist) * 100
print(f"  Diversity improvement: {diversity_improvement:+.1f}%")
print(f"  The diversity-aware engine maintains genetic diversity")
print(f"  while achieving comparable fitness optimization.")


STANDARD ENGINE vs DIVERSITY-AWARE ENGINE COMPARISON
Gen 1: Best Fitness = 63.0000
Gen 2: Best Fitness = 70.0000
Gen 3: Best Fitness = 72.0000
Gen 4: Best Fitness = 75.0000
Gen 5: Best Fitness = 79.0000
Gen 6: Best Fitness = 81.0000
Gen 7: Best Fitness = 81.0000
Gen 8: Best Fitness = 85.0000
Gen 9: Best Fitness = 86.0000
Gen 10: Best Fitness = 87.0000
Gen 11: Best Fitness = 87.0000
Gen 12: Best Fitness = 90.0000
Gen 13: Best Fitness = 91.0000
Gen 14: Best Fitness = 93.0000
Gen 15: Best Fitness = 93.0000
Gen 16: Best Fitness = 93.0000
Gen 17: Best Fitness = 95.0000
Gen 18: Best Fitness = 96.0000
Gen 19: Best Fitness = 97.0000
Gen 20: Best Fitness = 97.0000
Gen 21: Best Fitness = 97.0000
Gen 22: Best Fitness = 97.0000
Gen 23: Best Fitness = 98.0000
Gen 24: Best Fitness = 99.0000
Gen 25: Best Fitness = 99.0000
Gen 26: Best Fitness = 100.0000
Gen 27: Best Fitness = 100.0000
Gen 28: Best Fitness = 100.0000
Gen 29: Best Fitness = 100.0000
Gen 30: Best Fitness = 100.0000
Gen 31: Best Fitness 